In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam

In [2]:
import nltk

In [3]:
mma = nltk.corpus.gutenberg.sents('austen-emma.txt')

In [4]:
from collections import Counter

In [5]:
vocab = Counter()

In [6]:
for sent in mma:
    for w in sent:
        vocab[w]+=1

In [7]:
i = 1 # index 0 will be kept for padding
word2index = {}
index2word = {}
for w in vocab:
    if vocab[w]>5:
        word2index[w] = i
        index2word[i] = w
        i+=1

In [8]:
class RNN_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_size = 256
        self.vocab_size = 2170
        self.emb_dim = 100
        self.emb = nn.Embedding(self.vocab_size, self.emb_dim, padding_idx=0)
        self.rnn = nn.RNN(self.emb_dim, self.hidden_size) # input_dimension, hidden_dimension
        self.lin = nn.Linear(self.hidden_size, self.vocab_size)
        self.relu = nn.ReLU()
    
    def forward(self, inp_seq):
        inp = self.emb(inp_seq)
        h_0 = torch.rand(1, self.hidden_size)
        all_hidden_states, last_hidden_state = self.rnn(inp, h_0)
        out = self.lin(all_hidden_states)
        out = self.relu(out)
        return out

In [9]:
model = RNN_model()

In [10]:
model.load_state_dict(torch.load('emma_lm_rnn.pt'))

<All keys matched successfully>

In [11]:
seq_len = 10
seq = [418] # index corresponding to 'But'
for _ in range(seq_len):
    out = model(torch.IntTensor(seq))
    #next_token = torch.argmax(out[], dim=1)
    #seq.append
    out = out[-1,:]
    next_token = torch.argmax(out) # greedy decoding
    seq.append(next_token.item())

In [12]:
' '.join([index2word[s] for s in seq])

'But , I was to be in the of her ,'

In [13]:
# sampling

In [14]:
import numpy as np

In [15]:
seq_len = 10
seq = [418] # index corresponding to 'But'
for _ in range(seq_len):
    out = model(torch.IntTensor(seq))
    #next_token = torch.argmax(out[], dim=1)
    #seq.append
    out = out[-1,:]
    out = torch.softmax(out, dim=-1)
    #print(torch.sum(out))
    #next_token = torch.argmax(out) # greedy decoding
    #seq.append(next_token.item())
    break

In [16]:
probs = out.detach().numpy()

In [17]:
probs

array([0.00029547, 0.001946  , 0.00029547, ..., 0.00029547, 0.00029547,
       0.00029547], dtype=float32)

In [18]:
np.random.choice(2170, p = probs)

7

In [19]:
count = Counter()
for _ in range(1000):
    x = np.random.choice(2170, p = probs)
    count[x]+=1

In [20]:
count.most_common(5)

[(7, 79), (4, 61), (115, 44), (83, 33), (34, 23)]

In [21]:
probs[115]

0.04669948

In [22]:
torch.topk(out,5)

torch.return_types.topk(
values=tensor([0.0750, 0.0603, 0.0467, 0.0386, 0.0233], grad_fn=<TopkBackward0>),
indices=tensor([  7,   4, 115,  83,  34]))

In [23]:
def sample(out):
    probs = out.detach().numpy()
    s = np.random.choice(2170, p = probs)
    return s

In [24]:
seq_len = 10
seq = [418] # index corresponding to 'But'
for _ in range(seq_len):
    out = model(torch.IntTensor(seq))
    #next_token = torch.argmax(out[], dim=1)
    #seq.append
    out = out[-1,:]
    out = torch.softmax(out, dim=-1)
    #next_token = torch.argmax(out) # greedy decoding
    next_token = sample(out)
    seq.append(next_token)

In [25]:
' '.join([index2word[s] for s in seq])

'But I true maid intercourse establishment , visits , than Mr'

In [26]:
def sample_topk(out, k=5):
    values, indices = torch.topk(out,5)
    #print(values)
    values = values/torch.sum(values)
    probs = values.detach().numpy()
    #print(probs)
    indices = indices.numpy()
    s = np.random.choice(indices, p = probs)
    return s

In [27]:
torch.topk(out, k=5)

torch.return_types.topk(
values=tensor([0.3383, 0.0562, 0.0484, 0.0319, 0.0315], grad_fn=<TopkBackward0>),
indices=tensor([115,  22,  13,  19,   7]))

In [28]:
sample_topk(out)

115

In [29]:
seq_len = 10
seq = [418] # index corresponding to 'But'
for _ in range(seq_len):
    out = model(torch.IntTensor(seq))
    #next_token = torch.argmax(out[], dim=1)
    #seq.append
    out = out[-1,:]
    out = torch.softmax(out, dim=-1)
    #next_token = torch.argmax(out) # greedy decoding
    next_token = sample_topk(out)
    seq.append(next_token)

In [30]:
' '.join([index2word[s] for s in seq])

'But she was so , and was the to , and'

In [31]:
def sample_topn(out, threshold=0.7):
    values, indices = torch.sort(out, descending=True)
    cummulative_prob = torch.cumsum(values, dim=-1)
    for i in range(out.shape[0]):
        if cummulative_prob[i]>threshold:
            break
    print(i, cummulative_prob[i])
    i+=1
    values, indices = values[:i], indices[:i]
    #print(values)
    values = values/torch.sum(values)
    probs = values.detach().numpy()
    #print(probs)
    indices = indices.numpy()
    s = np.random.choice(indices, p = probs)
    return s

In [32]:
sample_topn(out)

209 tensor(0.7000, grad_fn=<SelectBackward0>)


75

In [33]:
seq_len = 10
seq = [418] # index corresponding to 'But'
for _ in range(seq_len):
    out = model(torch.IntTensor(seq))
    #next_token = torch.argmax(out[], dim=1)
    #seq.append
    out = out[-1,:]
    out = torch.softmax(out, dim=-1)
    #next_token = torch.argmax(out) # greedy decoding
    next_token = sample_topn(out)
    seq.append(next_token)

1077 tensor(0.7002, grad_fn=<SelectBackward0>)
1221 tensor(0.7002, grad_fn=<SelectBackward0>)
866 tensor(0.7000, grad_fn=<SelectBackward0>)
1069 tensor(0.7001, grad_fn=<SelectBackward0>)
352 tensor(0.7001, grad_fn=<SelectBackward0>)
1277 tensor(0.7003, grad_fn=<SelectBackward0>)
580 tensor(0.7001, grad_fn=<SelectBackward0>)
802 tensor(0.7001, grad_fn=<SelectBackward0>)
1050 tensor(0.7001, grad_fn=<SelectBackward0>)
1206 tensor(0.7001, grad_fn=<SelectBackward0>)


In [34]:
' '.join([index2word[s] for s in seq])

'But I think , Jane , it was not turns ,'